<p class="lead">Unit 11 showed that changing the representation moved the model's error far more than changing the algorithm. This unit is about the representation itself: how to build features from what you know about the chemistry or the biology, how to decide which of them to keep, and how to compress hundreds or thousands of columns into a handful you can look at. Three facts organise everything that follows. First, feature <em>construction</em> is where domain knowledge enters a model, and it is the step no library can do for you. Second, feature <em>selection</em> is a model fitted to the data and must therefore live inside the cross-validation, and even then its output is unstable when there are more columns than rows. Third, dimensionality <em>reduction</em> is a lossy compression whose losses you must be able to name: principal component analysis preserves variance and distorts nothing linear, while t-SNE and UMAP preserve neighbourhoods and will happily draw clusters in data that have none. We work on three feature families with completely different statistics — a constructed panel of 32 molecular descriptors, 2,048 fingerprint bits, and a 500-gene expression slice where the columns outnumber the rows almost four to one.</p>

<div class="box"><h4>Resumen en español</h4>
<p>Esta unidad trata de la representación de los datos. Se construyen variables a partir del conocimiento del dominio (razones, transformaciones logarítmicas, interacciones, variables categóricas de alta cardinalidad mediante <em>hashing</em>); se comparan los tres tipos de selección de variables (filtro, envolvente y embebida) ejecutados siempre dentro de la validación cruzada, midiendo su estabilidad con el índice de Jaccard entre pliegues; se deriva el análisis de componentes principales a partir de la descomposición en valores singulares, con gráfico de sedimentación, cargas, biplot y error de reconstrucción; se comparan t-SNE y UMAP advirtiendo sobre los conglomerados falsos; y se demuestra por simulación la maldición de la dimensionalidad. Se usan descriptores y huellas moleculares de ROCK1 y un subconjunto de 500 genes de un estudio de periodontitis. El resultado central: una variable de identidad (el andamiaje codificado por <em>hashing</em>) suma 0,064 de \(R^2\) con partición aleatoria y resta 0,204 con partición por andamiaje.</p></div>

<div class="box key"><h4>Learning objectives</h4>
<ul class="objectives">
<li>Engineer features from domain knowledge — descriptors, ratios, transformations, interactions, high-cardinality encodings — and encode them without leakage.</li>
<li>Apply filter, wrapper and embedded selection (variance, correlation, mutual information, recursive elimination, L1, permutation importance) inside cross-validation, and report selection stability with the Jaccard index.</li>
<li>Explain why selection rarely improves a regularised model's accuracy and what it is actually for.</li>
<li>Derive PCA from the singular value decomposition and read a scree plot, a loadings matrix, a biplot and a reconstruction error curve.</li>
<li>Use t-SNE and UMAP for visualisation, state what each preserves, and recognise a false cluster.</li>
<li>Explain the curse of dimensionality and demonstrate distance concentration by simulation.</li>
<li>Describe how fingerprints (sparse, binary, 2,048-dimensional), descriptors (dense, correlated) and expression matrices (\(p \gg n\)) differ statistically and what each demands of a method.</li>
</ul></div>

<div class="box bridge"><h4>How to read this unit</h4>
<p>Read linearly and run each chunk. This is the <strong>R edition</strong> of the unit; the <a href="../../Python/U13_Features_Dimensionality/index.html">Python version</a> runs the same experiments with scikit-learn. Three substitutions are forced by the R environment and are declared where they occur: the molecular descriptors are the eleven frozen RDKit columns of the curated table (there is no RDKit binding here, so the 45-descriptor panel of the Python edition becomes a 32-column panel <em>constructed</em> from those eleven — which is what Section 2 is about anyway); the fingerprints are loaded from a frozen sparse file; and mutual information, recursive feature elimination, feature hashing and permutation importance are written by hand, in six to ten lines each, because that is both shorter and clearer than finding an R package for each. <code>Rtsne</code><sup class="c"><a href="#r18">[18]</a></sup> and <code>umap</code><sup class="c"><a href="#r19">[19]</a></sup> do the embeddings, <code>glmnet</code><sup class="c"><a href="#r36">[36]</a></sup> the penalised fits, <code>ranger</code> the forests and <code>caret</code><sup class="c"><a href="#r35">[35]</a></sup> the resampling. In a fresh Colab R runtime run once: <code>install.packages(c("data.table","caret","ranger","glmnet","Rtsne","umap","cluster","digest","e1071","patchwork","R.utils"))</code>. <a href="../U11_Supervised_Workflow/index.html">Unit 11</a> (preprocessing chains and leakage) and <a href="../U12_Validation/index.html">Unit 12</a> (splits and validation) are prerequisites: every comparison here runs inside the schemes those units established.</p></div>



In [ ]:
## Setup: libraries, global seed, plotting defaults, data loader.
library(data.table); library(Matrix);  library(ggplot2); library(patchwork)
library(caret);      library(ranger);  library(glmnet);  library(digest)
library(e1071);      library(cluster); library(Rtsne);   library(umap)
set.seed(0)
theme_set(theme_minimal(base_size = 9) +
          theme(plot.title = element_text(face = "bold", size = 10),
                panel.grid.minor = element_blank()))
PAL <- c(blue = "#2b5d9b", orange = "#d2691e", green = "#3a7d44", purple = "#6a4c93")
REPO <- "DanielBustosG/DanielBustosG.github.io"   # GitHub repository hosting this course (set once in common/config.json)
DATA_URL <- paste0("https://raw.githubusercontent.com/", REPO, "/main/DataScience/data/")
data_file <- function(name) {   # local ../data/<name>; in Colab, downloaded once into data/
  local <- file.path("..", "data", name)
  if (file.exists(local)) return(local)
  dest <- file.path("data", name)
  if (!file.exists(dest)) {
    dir.create("data", showWarnings = FALSE)
    download.file(paste0(DATA_URL, name), dest, quiet = TRUE, mode = "wb")
  }
  dest
}
data_path <- function(name) data_file(name)
load_data  <- function(name, ...) data.table::fread(data_path(name), ...)
cat("R", as.character(getRversion()), "| caret", as.character(packageVersion("caret")),
    "| glmnet", as.character(packageVersion("glmnet")), "| Rtsne", as.character(packageVersion("Rtsne")),
    "| umap", as.character(packageVersion("umap")), "\n")


In [ ]:
## Three feature families: a constructed descriptor panel, 2048 fingerprint bits, 500 genes.
df  <- load_data("rock1_chembl_curated.csv")
reg <- df[censored == FALSE]
y        <- reg$pActivity
scaffold <- reg$scaffold_smiles
BASE <- c("MolWt", "MolLogP", "TPSA", "HBD", "HBA", "RotB",
          "AromaticRings", "RingCount", "HeavyAtoms", "FractionCSP3", "QED")
B <- as.matrix(reg[, ..BASE]); d <- as.data.table(B)
fpl <- load_data("rock1_morgan_sparse.csv.gz"); ikey <- reg$inchikey; fpl <- fpl[id %in% ikey]
FP  <- as.matrix(sparseMatrix(i = match(fpl$id, ikey), j = fpl$bit + 1L, x = 1,
                              dims = c(length(ikey), 2048), dimnames = list(NULL, paste0("bit", 0:2047))))
ex  <- load_data("geo_periodontitis_expr.csv")
E_all <- as.matrix(ex[, -1]); rownames(E_all) <- ex$symbol
top500 <- order(apply(E_all, 1, var), decreasing = TRUE)[1:500]
E <- t(E_all[top500, ]); gene_names <- ex$symbol[top500]
meta <- load_data("geo_periodontitis_meta.csv")
E_label <- as.integer(meta$status[match(rownames(E), meta$sample_id)] == "diseased")
cat(sprintf("base descriptors %5d x %4d  (dense, continuous)\n", nrow(B), ncol(B)))
cat(sprintf("fingerprints     %5d x %4d  (sparse binary, %.1f%% of entries are 1)\n",
            nrow(FP), ncol(FP), 100 * mean(FP)))
cat(sprintf("expression       %5d x %4d  (p >> n: %d samples, %d genes; %d diseased)\n",
            nrow(E), ncol(E), nrow(E), ncol(E), sum(E_label)))
cat("missing values: descriptors", sum(is.na(B)), "| expression", sum(is.na(E)), "\n")



## 1. A feature matrix is a scientific claim

Choosing features is choosing what the model is allowed to notice. A molecular weight column asserts that size matters; a fingerprint bit asserts that the presence of one particular substructure matters; a gene column asserts that the expression of that gene matters. The three families loaded above differ not only in size but in their whole statistical character, and almost every practical difficulty in this unit follows from those differences.

<div class="box bridge"><h4>Domain context — descriptors, fingerprints and expression matrices</h4>
<p>A <strong>molecular descriptor</strong> is a number computed from a structure that means something physical: molecular weight, the calculated oil/water partition coefficient MolLogP (greasiness), the topological polar surface area TPSA (how much polar surface the molecule exposes), counts of rings, rotatable bonds and hydrogen-bond donors and acceptors, the fraction of sp³ (three-dimensional rather than flat) carbons, and the composite drug-likeness score QED<sup class="c"><a href="#r28">[28]</a></sup>. Descriptors are dense, continuous and heavily correlated, because most of them are partly measures of size. A <strong>fingerprint</strong> is a long binary vector in which bit <em>i</em> records whether a particular circular substructure is present<sup class="c"><a href="#r27">[27]</a></sup>; it is sparse (about 2.6% of bits set here), each column is almost meaningless alone, and the useful information is in combinations. A <strong>gene-expression matrix</strong> from a microarray or RNA-seq experiment has one column per gene (tens of thousands) and one row per sample (tens to hundreds): the <em>p ≫ n</em> regime, where almost any pattern can be fitted and nothing can be trusted without stability analysis. Our expression slice comes from gingival tissue of patients with periodontitis — a chronic inflammatory disease of the tooth-supporting tissues — in which each patient contributed one diseased and one healthy site<sup class="c"><a href="#r30">[30]</a></sup><sup class="c"><a href="#r31">[31]</a></sup>.</p></div>

The chunk below builds the constructed panel that Sections 2 to 4 use, and then measures the statistical signature of each family.



In [ ]:
## The constructed 32-column panel: 11 frozen descriptors + 6 domain ratios + 15 transforms/interactions.
ENG <- data.table(
  polar_density  = d$TPSA / d$MolWt,                        ## polar surface per unit mass
  logp_per_heavy = d$MolLogP / d$HeavyAtoms,                ## lipophilic efficiency per atom
  hbd_hba_ratio  = (d$HBD + 1) / (d$HBA + 1),
  aromatic_frac  = d$AromaticRings / pmax(d$RingCount, 1),
  flexibility    = d$RotB / pmax(d$HeavyAtoms, 1),
  log_MolWt      = log(d$MolWt))
EXT <- data.table(
  sqrt_TPSA      = sqrt(d$TPSA),            log1p_RotB    = log1p(d$RotB),
  logp_sq        = d$MolLogP^2,             MolWt_per_ring = d$MolWt / pmax(d$RingCount, 1),
  HBA_per_heavy  = d$HBA / d$HeavyAtoms,    HBD_per_heavy = d$HBD / d$HeavyAtoms,
  ring_density   = d$RingCount / d$HeavyAtoms, logp_x_tpsa = d$MolLogP * d$TPSA,
  molwt_x_csp3   = d$MolWt * d$FractionCSP3,   qed_x_logp  = d$QED * d$MolLogP,
  arom_x_rotb    = d$AromaticRings * d$RotB,   hbd_x_hba   = d$HBD * d$HBA,
  csp3_x_rings   = d$FractionCSP3 * d$RingCount, tpsa_per_heavy = d$TPSA / d$HeavyAtoms,
  heavy_sq       = d$HeavyAtoms^2)
P <- as.matrix(cbind(d, ENG, EXT))
pair_corr <- function(M, max_cols = 400, seed = 0) {
  keep <- which(apply(M, 2, sd) > 0)                   ## constant columns have no correlation
  set.seed(seed); cols <- sample(keep, min(max_cols, length(keep)))
  C <- cor(M[, cols]); abs(C[upper.tri(C)])
}
nz <- function(M) M[, apply(M, 2, sd) > 0, drop = FALSE]
stats_tbl <- data.table(
  family = c("constructed panel (32)", "fingerprints (2048)", "expression (500 genes)"),
  rows = c(nrow(P), nrow(FP), nrow(E)), columns = c(ncol(P), ncol(FP), ncol(E)),
  rows_per_column = c(nrow(P) / ncol(P), nrow(FP) / ncol(FP), nrow(E) / ncol(E)),
  fraction_nonzero = c(1, mean(FP), 1),
  median_abs_corr = c(median(pair_corr(P)), median(pair_corr(FP)), median(pair_corr(E))),
  max_abs_skew = c(max(abs(apply(P, 2, skewness))), max(abs(apply(nz(FP), 2, skewness))),
                   max(abs(apply(E, 2, skewness)))))
knitr::kable(stats_tbl[, lapply(.SD, function(v) if (is.numeric(v)) round(v, 3) else v)],
             caption = "The statistical signature of the three feature families.")



<p class="caption"><b>Table (Section 1).</b> The three families compared. The constructed panel has 86 rows per column — comfortable for any method — a median absolute pairwise correlation of 0.267, which is very high (most columns partly measure size), and a worst skewness of 6.8 (the <code>hbd_x_hba</code> interaction, a product of two counts bounded below by zero). The fingerprints have only 1.35 rows per column, 2.6% of their entries set, and a median pairwise correlation of 0.006 over the non-constant bits, because most bit pairs never co-occur; their worst skewness is 52.5, as a bit set in a handful of molecules must be. The expression slice has 0.276 rows per column: fewer samples than genes, the regime in which a linear model can fit the training data exactly and learn nothing. Every method in this unit behaves differently on these three matrices, and the rows-per-column ratio is the best single predictor of how much trouble you are in.</p>

## 2. Constructing features from domain knowledge

### 2.1 Ratios, densities and transformations

Medicinal chemistry has a vocabulary of *derived* quantities that are more informative than their parts: polar surface per unit mass, lipophilicity per heavy atom (the basis of ligand-efficiency metrics), the ratio of aromatic to total rings, flexibility per atom. These are exactly the interaction and ratio features a model with limited capacity cannot invent for itself, because a linear model can add columns but cannot divide them.

The comparison below is the honest test of whether they earn their place. Note the last row: instead of the Python edition's 45-descriptor RDKit panel, which R cannot compute here, it adds *new information* in the only form available — sixty principal components of the fingerprint matrix.



In [ ]:
## Do the engineered features earn their place? Four matrices, two models, one 5-fold CV. (~15 s)
r2   <- function(o, p) 1 - sum((o - p)^2) / sum((o - mean(o))^2)
rmse <- function(o, p) sqrt(mean((o - p)^2))
zscale <- function(Xtr) { m <- colMeans(Xtr); s <- apply(Xtr, 2, sd); s[s == 0] <- 1; list(m = m, s = s) }
f_ridge <- function(Xtr, yy, Xte, lam = 0.5) {            ## scaling fitted on the training fold only
  z <- zscale(Xtr)
  if (ncol(Xtr) < 2) { fit <- lm(yy ~ Xtr); return(as.numeric(cbind(1, Xte) %*% coef(fit))) }
  m <- glmnet(scale(Xtr, z$m, z$s), yy, alpha = 0, lambda = lam)
  as.numeric(predict(m, scale(Xte, z$m, z$s)))
}
f_rf <- function(Xtr, yy, Xte) {
  m <- ranger(x = Xtr, y = yy, num.trees = 200, seed = 1, num.threads = 4)
  predict(m, Xte)$predictions
}
set.seed(0); folds <- split(sample(seq_along(y)), rep(1:5, length.out = length(y)))
cvR2 <- function(X, fp = f_ridge) mean(sapply(folds, function(te) {
  tr <- setdiff(seq_along(y), te); r2(y[te], fp(X[tr, , drop = FALSE], y[tr], X[te, , drop = FALSE]))
}))
pfp <- prcomp(FP, center = TRUE, scale. = FALSE, rank. = 60)   ## reused in Sections 4 and 5
sets <- list("11 base descriptors" = B, "11 base + 6 engineered" = cbind(B, as.matrix(ENG)),
             "32-column constructed panel" = P, "11 base + 60 fingerprint PCs" = cbind(B, pfp$x))
eng_tbl <- rbindlist(lapply(names(sets), function(nm)
  data.table(features = nm, p = ncol(sets[[nm]]),
             ridge_R2 = cvR2(sets[[nm]]), forest_R2 = cvR2(sets[[nm]], f_rf))))
knitr::kable(eng_tbl[, lapply(.SD, function(v) if (is.numeric(v)) round(v, 3) else v)],
             caption = "Hand-built features against added information.")



<p class="caption"><b>Table (Section 2.1).</b> Six hand-built ratio features added to eleven basic descriptors raise the ridge \(R^2\) from 0.098 to 0.115 and leave the random forest unchanged (0.344 to 0.337 — a change smaller than the fold-to-fold noise). Adding nine more transforms and interactions to reach the 32-column panel gets ridge to 0.128 and the forest to 0.348. Replacing all of that with <em>new information</em> — the same eleven descriptors plus sixty fingerprint principal components — lifts ridge to 0.359 and the forest to 0.508. The lesson is uncomfortable and worth absorbing: a tree ensemble can already construct ratios and interactions implicitly by splitting on both parents, so hand-crafted ratios buy it almost nothing, while for a linear model they help only when the ratio really is the mechanism. Feature engineering pays when it <em>adds information</em> (more descriptors, a new assay, a structural feature) and rarely when it only rearranges information the model already has.</p>

### 2.2 Transformations: making a column usable

Skewed, heavy-tailed or zero-inflated columns violate the implicit assumptions of distance-based and linear methods. The classical remedies are the logarithm, the Box–Cox family<sup class="c"><a href="#r33">[33]</a></sup> and its zero-tolerant Yeo–Johnson variant, binning, and — for descriptors that are zero for most molecules — an indicator column plus the value. `caret::preProcess()` implements Yeo–Johnson as a *fitted* step, which is the only form in which it is safe to use.



In [ ]:
## Before and after: the three most skewed panel columns under a Yeo-Johnson power transform.
sk <- sort(abs(apply(P, 2, skewness)), decreasing = TRUE)
worst <- names(sk)[1:3]
pp_yj <- caret::preProcess(as.data.frame(P), method = c("YeoJohnson", "center", "scale"))
YJ <- as.matrix(predict(pp_yj, as.data.frame(P)))
long <- rbindlist(lapply(worst, function(nm) rbindlist(list(
  data.table(feature = nm, stage = sprintf("raw (skew %.1f)", skewness(P[, nm])), value = P[, nm]),
  data.table(feature = nm, stage = sprintf("Yeo-Johnson (skew %.2f)", skewness(YJ[, nm])), value = YJ[, nm])))))
print(ggplot(long, aes(value, fill = stage)) + geom_histogram(bins = 40, show.legend = FALSE) +
  facet_wrap(~ feature + stage, scales = "free", ncol = 2) +
  scale_fill_manual(values = rep(unname(PAL[c("orange", "blue")]), 3)) +
  labs(x = NULL, y = "compounds", title = "Symmetrising the three most skewed columns") +
  theme(strip.text = element_text(size = 7.5), axis.text.y = element_blank()))
QN <- apply(P, 2, function(v) qnorm(rank(v, ties.method = "average") / (length(v) + 1)))
cat("most skewed columns:", paste(names(sk)[1:5], round(sk[1:5], 1), sep = "=", collapse = "  "), "\n")
cat(sprintf("ridge R2: raw + standardise %.3f | Yeo-Johnson %.3f | quantile-normal %.3f\n",
            cvR2(P), cvR2(YJ), cvR2(QN)))



<p class="caption"><b>Figure 1.</b> The three most skewed panel columns before (orange) and after (blue) a Yeo–Johnson power transform. Raw skewness of 2.5 to 6.8 — products and ratios of counts bounded below by zero, with long right tails — becomes 0.01 to 0.09. The cross-validated effect on ridge regression, printed below the figure, is almost nil: \(R^2\) moves from 0.128 to 0.127 with Yeo–Johnson and to 0.125 with a quantile transform. That is the honest result and it is worth keeping: symmetrising a predictor helps distance-based methods, visual inspection and coefficient interpretation, but a regularised linear model on 32 columns does not need it. Note that both transforms are <em>fitted</em> (they learn an exponent or a set of quantiles), so both belong inside the resampling — a transform fitted on the full dataset is item one of Unit 11's leakage catalogue. The quantile transform above is deliberately computed on all rows to keep the code short, which is exactly why its number should be read as an illustration and not as a performance estimate.</p>

### 2.3 High-cardinality categoricals and feature hashing

Some columns are categorical with hundreds or thousands of levels: the scaffold of a molecule (1,218 distinct values here), a target gene, an assay identifier, a hospital. One-hot encoding them adds a column per level, most of which are seen once. **Feature hashing** maps each level through a hash function into a fixed number of buckets<sup class="c"><a href="#r29">[29]</a></sup>, trading collisions for a bounded, fixed-width representation that needs no vocabulary and handles unseen levels gracefully. R has no `FeatureHasher`, but `digest` gives a hash function and the rest is one line.



In [ ]:
## Scaffold identity as a hashed categorical feature: 1,218 levels into 256 buckets.
hash_bucket <- function(s, n = 256) as.integer(strtoi(substr(digest(s, algo = "xxhash32"), 1, 7), 16L) %% n)
us <- unique(scaffold); bk <- vapply(us, hash_bucket, integer(1))
bidx <- bk[match(scaffold, us)]
H  <- as.matrix(sparseMatrix(i = seq_along(bidx), j = bidx + 1L, x = 1, dims = c(length(bidx), 256)))
Xh <- cbind(B, H)
tb <- table(bk)
cat("distinct scaffolds:", length(us), " -> one-hot would add", length(us), "columns; hashing adds 256\n")
cat("buckets actually used:", length(tb), "| most crowded bucket holds", max(tb),
    "scaffolds | scaffolds sharing a bucket with another:", sum(tb[as.character(bk)] > 1), "\n")
cat(sprintf("ridge R2 with 11 descriptors                   : %.3f\n", cvR2(B)))
cat(sprintf("ridge R2 with 11 descriptors + hashed scaffold : %.3f\n", cvR2(Xh)))



All 256 buckets are used, the most crowded holds 13 scaffolds, and 1,203 of the 1,218 scaffolds share their bucket with at least one other — collisions are the rule, not the exception, at this compression ratio. Adding the hashed scaffold lifts \(R^2\) from 0.098 to 0.162 — apparently the best feature-engineering result of the unit so far. It is also a trap, and the next chunk springs it.

### 2.4 A failure mode: a feature that only works under the wrong split

Scaffold identity tells the model which series a compound belongs to. Under a random split, most test compounds share their scaffold with training compounds, so the feature acts as a lookup table of series averages — genuinely predictive, and genuinely useless for the question "how potent is a compound from a series we have not worked on?". Under the scaffold-grouped split of Unit 12 the lookup table has no entry, and the 256 extra columns become pure noise for the regulariser to fight.



In [ ]:
## The same feature, judged by two splits.
gk <- caret::groupKFold(scaffold, k = 5)
cvR2g <- function(X, fp = f_ridge) mean(sapply(gk, function(tr) {
  te <- setdiff(seq_along(y), tr); r2(y[te], fp(X[tr, , drop = FALSE], y[tr], X[te, , drop = FALSE]))
}))
gain_tbl <- data.table(
  protocol = c("random 5-fold", "scaffold-grouped 5-fold"),
  descriptors_only = c(cvR2(B), cvR2g(B)),
  plus_hashed_scaffold = c(cvR2(Xh), cvR2g(Xh)))
gain_tbl[, gain := plus_hashed_scaffold - descriptors_only]
print(ggplot(melt(gain_tbl[, .(protocol, descriptors_only, plus_hashed_scaffold)], id.vars = "protocol"),
       aes(protocol, value, fill = variable)) +
  geom_hline(yintercept = 0, linewidth = 0.4) +
  geom_col(position = "dodge", width = 0.6) +
  geom_text(data = gain_tbl, aes(protocol, pmax(descriptors_only, plus_hashed_scaffold) + 0.02,
                                 label = sprintf("%+.3f", gain)), inherit.aes = FALSE, size = 3.2) +
  scale_fill_manual(values = c(descriptors_only = "grey60", plus_hashed_scaffold = unname(PAL["blue"])),
                    labels = c("11 descriptors", "+ hashed scaffold (256)"), name = NULL) +
  labs(x = NULL, y = expression(cross-validated~R^2),
       title = "The same engineered feature, two validation protocols") + theme(legend.position = "top"))
knitr::kable(gain_tbl[, lapply(.SD, function(v) if (is.numeric(v)) round(v, 3) else v)],
             caption = "An identity feature under two protocols.")



<p class="caption"><b>Figure 2 and table.</b> The hashed-scaffold feature adds +0.064 \(R^2\) under a random split and <em>subtracts</em> 0.204 under a scaffold-grouped split, where it turns a weak model (\(R^2\) = 0.025) into a harmful one (\(R^2\) = −0.179, far worse than predicting the mean). Nothing changed except the question being asked. This is the most important practical lesson of the unit: <strong>every feature-engineering decision must be evaluated under the split that matches the deployment claim</strong>, because identity-like features (series, patient, batch, site, assay) are reliable predictors of the training set and silent on anything new. The effect is much larger here than in the Python edition (+0.08 and −0.05 there) because our hash is built on exactly the scaffold used to group the folds, so the 256 columns carry no usable signal at all on a grouped test fold. The clinical analogue is adding <code>patient_id</code> or <code>hospital</code> as a feature, which improves every random-split metric and predicts nothing about the next patient.</p>

<div class="exercise"><h4>Exercise 1</h4>
<p>Replace the hashed scaffold by a <em>target-encoded</em> scaffold: each compound's scaffold replaced by the mean pActivity of that scaffold, computed inside each training fold only (unseen scaffolds get the training mean). Measure it under both protocols. Then compute it once on the whole dataset (the wrong way) and measure again; report the three numbers and say which of the two errors — wrong encoding or wrong split — costs more.</p></div>

## 3. Feature selection

Selection methods come in three families, distinguished by how much they know about the model<sup class="c"><a href="#r1">[1]</a></sup><sup class="c"><a href="#r2">[2]</a></sup>.

- **Filters** score each feature independently of any model: variance, absolute correlation with the target, the univariate F statistic, mutual information (which detects non-linear dependence). Cheap, model-agnostic, and blind to redundancy — two perfectly correlated useful features both score highly and both get selected.
- **Wrappers** use the model itself: recursive feature elimination fits, ranks, drops the weakest, and repeats<sup class="c"><a href="#r6">[6]</a></sup>. Expensive (one fit per elimination round), aware of interactions, and prone to overfitting the selection when data are scarce.
- **Embedded** methods select as part of fitting: the L1 penalty of the lasso drives coefficients exactly to zero<sup class="c"><a href="#r7">[7]</a></sup>, the elastic net does the same while keeping correlated groups together<sup class="c"><a href="#r8">[8]</a></sup>, and tree ensembles report impurity-based importances.

Whatever the family, the selection must be **refitted inside every training fold**: selecting on the whole dataset and then cross-validating is the leak that turns noise into apparent signal<sup class="c"><a href="#r3">[3]</a></sup> (Unit 11 measured it). The chunk below writes each selector as a function of the training fold only, and `cvR2()` calls it inside the loop.



In [ ]:
## Four selectors, each a function of (X_train, y_train) returning column indices.
mi_score <- function(X, yy, bins = 10) {         ## mutual information from its definition, on quantile bins
  yb <- cut(yy, quantile(yy, 0:bins / bins), include.lowest = TRUE, labels = FALSE)
  apply(X, 2, function(v) {
    if (sd(v) == 0) return(0)
    q  <- unique(quantile(v, 0:bins / bins)); xb <- cut(v, q, include.lowest = TRUE, labels = FALSE)
    tab <- table(xb, yb) / length(yy); px <- rowSums(tab); py <- colSums(tab)
    sum(tab * log((tab + 1e-12) / (outer(px, py) + 1e-12)))
  })
}
rfe_select <- function(X, yy, k = 20, step = 2) { ## recursive elimination with a ridge ranker
  keep <- seq_len(ncol(X))
  while (length(keep) > k) {
    z <- zscale(X[, keep, drop = FALSE])
    m <- glmnet(scale(X[, keep, drop = FALSE], z$m, z$s), yy, alpha = 0, lambda = 0.5)
    b <- abs(as.numeric(coef(m))[-1])
    keep <- keep[-order(b)[seq_len(min(step, length(keep) - k))]]
  }
  keep
}
lasso_select <- function(X, yy, lam = 0.03) {
  z <- zscale(X); m <- glmnet(scale(X, z$m, z$s), yy, alpha = 1, lambda = lam)
  which(as.numeric(coef(m))[-1] != 0)
}
sel_methods <- list(
  "top-20 |correlation|"      = function(X, yy) order(abs(as.numeric(cor(X, yy))), decreasing = TRUE)[1:20],
  "top-20 mutual information" = function(X, yy) order(mi_score(X, yy), decreasing = TRUE)[1:20],
  "RFE to 20 (ridge)"         = function(X, yy) rfe_select(X, yy, 20),
  "L1 (lasso) selected"       = function(X, yy) lasso_select(X, yy))
with_selection <- function(sel) function(Xtr, yy, Xte) {
  k <- sel(Xtr, yy); f_ridge(Xtr[, k, drop = FALSE], yy, Xte[, k, drop = FALSE])
}
vt <- which(apply(P, 2, var) > 0.01 * mean(apply(P, 2, var)))
sel_tbl <- rbindlist(c(
  list(data.table(strategy = "all 32 panel columns", kept = ncol(P), R2 = cvR2(P))),
  lapply(names(sel_methods), function(nm)
    data.table(strategy = nm, kept = length(sel_methods[[nm]](P, y)), R2 = cvR2(P, with_selection(sel_methods[[nm]])))),
  list(data.table(strategy = "variance filter", kept = length(vt), R2 = cvR2(P[, vt, drop = FALSE])))))
knitr::kable(sel_tbl[, .(strategy, kept, R2 = round(R2, 3))],
             caption = "Six protocols, one ridge model, selection refitted inside every fold.")



<p class="caption"><b>Table (Section 3).</b> Six protocols, one model. Keeping all 32 panel columns gives the best cross-validated \(R^2\) (0.128); every selection method <em>loses</em> accuracy — mutual information most (0.108), the correlation filter and the lasso next (0.118 and 0.119), recursive elimination almost nothing (0.127) — and the crude variance filter, which keeps only the 9 columns whose variance exceeds a hundredth of the mean variance, is a disaster (0.055). The ranking is not an accident: ridge regression already handles redundant columns by shrinking them, so removing columns can only remove information. Selection is therefore not an accuracy tool here. It is a tool for <em>interpretability</em> (a model with 20 named descriptors can be discussed with a chemist), for <em>cost</em> (fewer descriptors to compute, fewer genes to assay in a clinical panel), and for <em>stability</em> of the scientific story. When a paper reports that selection improved performance, ask whether the selection was refitted inside the folds.</p>



In [ ]:
## Which features does each method choose, and do they agree? (Jaccard overlap across folds)
jaccard <- function(sets) {
  pairs <- combn(length(sets), 2)
  mean(apply(pairs, 2, function(ij)
    length(intersect(sets[[ij[1]]], sets[[ij[2]]])) / length(union(sets[[ij[1]]], sets[[ij[2]]]))))
}
per_fold_sets <- function(sel) lapply(folds, function(te) {
  tr <- setdiff(seq_along(y), te); sel(P[tr, , drop = FALSE], y[tr])
})
for (nm in names(sel_methods)) {
  s <- per_fold_sets(sel_methods[[nm]])
  cat(sprintf("%-26s Jaccard %.3f | sizes %s | selected in all 5 folds: %d\n",
              nm, jaccard(s), paste(lengths(s), collapse = ","), length(Reduce(intersect, s))))
}
core <- Reduce(intersect, per_fold_sets(sel_methods[[1]]))
cat("\nthe", length(core), "panel columns chosen by the correlation filter in every fold:\n")
cat(paste(colnames(P)[sort(core)], collapse = ", "), "\n")



With 2,764 rows and 32 columns the selections are stable: the mean Jaccard overlap between folds is 0.906 for the correlation filter, 0.804 for mutual information, 0.828 for recursive elimination and 0.731 for the lasso, and 18, 15, 16 and 6 of the chosen columns respectively are the same in all five folds. The lasso is the least stable by that measure, partly because it is the only method whose *number* of selections varies (8 to 11 columns across folds) — Jaccard punishes that automatically. The printed core set — molecular weight and heavy-atom count, the hydrogen-bond and rotatable-bond counts, the aromatic and total ring counts, the sp³ fraction and the ratios built from them — is the part of the story a chemist can be told.

### 3.1 Permutation importance

Impurity-based importances from trees are biased towards high-cardinality and continuous features<sup class="c"><a href="#r10">[10]</a></sup>. **Permutation importance** avoids that by measuring the loss of performance when one column is shuffled<sup class="c"><a href="#r9">[9]</a></sup>, which is model-agnostic and directly interpretable — but must be computed on *held-out* data, otherwise it reports what the model memorised rather than what it uses. Five lines of R, and the five lines are the definition.



In [ ]:
## Permutation importance on held-out data, next to the forest's impurity importance. (~20 s)
set.seed(0); te_i <- sample(seq_along(y), round(0.25 * length(y))); tr_i <- setdiff(seq_along(y), te_i)
rf_imp <- ranger(x = P[tr_i, ], y = y[tr_i], num.trees = 300, seed = 1, num.threads = 4,
                 importance = "impurity")
base_r2 <- r2(y[te_i], predict(rf_imp, P[te_i, ])$predictions)
set.seed(7)
perm <- sapply(seq_len(ncol(P)), function(j) mean(replicate(5, {
  Xp <- P[te_i, ]; Xp[, j] <- sample(Xp[, j])             ## break the link to the target, keep the margin
  base_r2 - r2(y[te_i], predict(rf_imp, Xp)$predictions)
})))
imp <- data.table(feature = colnames(P), impurity = rf_imp$variable.importance / sum(rf_imp$variable.importance),
                  permutation = perm)[order(-permutation)]
top <- imp[1:12][order(permutation)]
print(ggplot(melt(top, id.vars = "feature"), aes(factor(feature, levels = top$feature), value, fill = variable)) +
  geom_col(position = "dodge", width = 0.72) + coord_flip() +
  scale_fill_manual(values = c(impurity = "grey60", permutation = unname(PAL["blue"])),
                    labels = c("impurity (training)", expression(permutation~(held-out~R^2~lost))), name = NULL) +
  labs(x = NULL, y = "importance", title = "Two notions of importance for the same forest") +
  theme(legend.position = "top", axis.text.y = element_text(size = 7.5)))
cat(sprintf("held-out R2 of the forest: %.3f\n", base_r2))
cat(sprintf("sum of impurity importances = %.2f (always 1) | sum of permutation importances = %.3f\n",
            sum(imp$impurity), sum(imp$permutation)))
print(imp[1:6][, .(feature, impurity = round(impurity, 4), permutation = round(permutation, 4))])



<p class="caption"><b>Figure 3.</b> The twelve most important panel columns by permutation importance (blue, measured on 691 held-out compounds) with the forest's impurity importance for comparison (grey). The two rankings broadly agree at the top but not in detail, and their scales are different by construction: impurity importances are a decomposition of the training fit and sum to 1, while permutation importances are \(R^2\) units lost on unseen data and here sum to 0.262 — less than the model's held-out \(R^2\) of 0.338, because correlated columns share credit. Shuffling one of two near-duplicate columns leaves the other to carry the signal, so both can look unimportant. The single most important column by both measures is <code>flexibility</code> (rotatable bonds per heavy atom), an engineered ratio, which is the one unambiguous success of Section 2.1. With feature blocks as correlated as these, importances are a guide to the story, never a proof of mechanism; Unit 16 treats the problem properly.</p>

### 3.2 The p ≫ n regime: selection on gene expression

Everything above was comfortable because there were eighty-six rows per column. The expression slice has 138 samples and 500 genes. The same filter, the same code, a completely different outcome.



In [ ]:
## The same selection protocol on 138 samples x 500 genes.
set.seed(0); gf <- caret::createFolds(factor(E_label), k = 5)
t_stat <- function(X, lab) apply(X, 2, function(v) abs(as.numeric(t.test(v ~ lab)$statistic)))
auc_roc <- function(o, p) { r <- rank(p); n1 <- sum(o == 1); n0 <- sum(o == 0)
  (sum(r[o == 1]) - n1 * (n1 + 1) / 2) / (n1 * n0) }
gsets <- lapply(gf, function(te) {
  tr <- setdiff(seq_len(nrow(E)), te); order(t_stat(E[tr, ], E_label[tr]), decreasing = TRUE)[1:20]
})
gauc <- sapply(gf, function(te) {
  tr <- setdiff(seq_len(nrow(E)), te)
  k  <- order(t_stat(E[tr, ], E_label[tr]), decreasing = TRUE)[1:20]
  z  <- zscale(E[tr, k]); m <- glmnet(scale(E[tr, k], z$m, z$s), E_label[tr], family = "binomial", lambda = 0.01)
  auc_roc(E_label[te], as.numeric(predict(m, scale(E[te, k, drop = FALSE], z$m, z$s), type = "response")))
})
freq <- table(unlist(gsets))
cnt  <- table(as.integer(freq))
print(ggplot(data.frame(folds = as.integer(names(cnt)), genes = as.integer(cnt)), aes(folds, genes)) +
  geom_col(fill = PAL[["blue"]], width = 0.6) +
  geom_text(aes(label = genes), vjust = -0.3, size = 3) +
  scale_x_continuous(breaks = 1:5) +
  labs(x = "number of folds (out of 5) in which the gene was selected", y = "genes",
       title = "Instability of a 20-gene signature (138 samples, 500 genes)"))
cat(sprintf("panel  (2764 x 32) : mean Jaccard %.3f\n", jaccard(per_fold_sets(sel_methods[[1]]))))
cat(sprintf("genes  (138 x 500) : mean Jaccard %.3f | selected in all 5 folds: %d of 20 | ever selected: %d\n",
            jaccard(gsets), length(Reduce(intersect, gsets)), length(freq)))
cat(sprintf("classification ROC-AUC with selection inside the CV: %.3f (folds %s)\n",
            mean(gauc), paste(round(gauc, 2), collapse = " ")))
cat("most frequently selected genes:",
    paste(gene_names[as.integer(names(sort(freq, decreasing = TRUE)))[1:6]], collapse = ", "), "\n")



<p class="caption"><b>Figure 4.</b> How often each gene enters the 20-gene signature across five folds. Only 4 genes are selected in all five folds, while 16 appear exactly once and 43 genes in total enter the signature somewhere: the mean Jaccard overlap between folds is 0.354, against 0.906 for the molecular panel. The classifier nevertheless achieves a cross-validated ROC-AUC of 0.909 — with a startling fold-to-fold range of 0.79 to 0.99 on 27 test samples — which is the paradox of <em>p ≫ n</em> data: the <strong>signature is unstable but the prediction is good</strong>, because hundreds of genes carry the same inflammatory signal and any twenty of them suffice. A paper that names its twenty genes as "the signature" is over-claiming; the honest report is the selection frequency, or a stability-selection procedure that re-samples the data many times and keeps only features selected often<sup class="c"><a href="#r4">[4]</a></sup><sup class="c"><a href="#r5">[5]</a></sup>. Batch structure makes this worse, since technical covariates produce exactly the kind of correlated block that selection cannot distinguish from biology<sup class="c"><a href="#r32">[32]</a></sup>.</p>

<div class="box warn"><h4>Common misconception</h4>
<p>"My selected features are the important biology." They are <em>a</em> sufficient subset found by one algorithm on one sample. Report how often each feature is selected across resamples, and treat any single list as a hypothesis — especially when rows are fewer than columns.</p></div>

## 4. Principal component analysis, from the singular value decomposition

### 4.1 The derivation and a worked example by hand

Centre the data matrix \(X\) (\(n \times p\)) so each column has mean zero and write its singular value decomposition \(X = U S V^{\top}\). The columns of \(V\) are the **principal directions**, the matrix \(T = XV = US\) holds the **scores** (the coordinates of each sample in the new basis), and the variance explained by component \(j\) is \(s_j^2/(n-1)\), where \(s_j\) is the \(j\)-th singular value. Equivalently, \(V\) holds the eigenvectors of the covariance matrix \(X^{\top}X/(n-1)\) with eigenvalues \(s_j^2/(n-1)\): PCA *is* the SVD, which is why `prcomp()` computes it that way (and why `irlba` makes it feasible on huge matrices through a randomised partial SVD<sup class="c"><a href="#r15">[15]</a></sup>). Historically the construction goes back to Pearson<sup class="c"><a href="#r12">[12]</a></sup> and Hotelling<sup class="c"><a href="#r13">[13]</a></sup>, and the optimality of the truncated SVD as a low-rank approximation to Eckart and Young<sup class="c"><a href="#r14">[14]</a></sup>.

A two-dimensional example can be done on paper. Take the standardised values of two correlated descriptors; the covariance matrix of two standardised columns is \(\begin{pmatrix} 1 & r \\ r & 1\end{pmatrix}\), whose eigenvalues are exactly \(1+r\) and \(1-r\) with eigenvectors \((1,1)/\sqrt{2}\) and \((1,-1)/\sqrt{2}\). So for two standardised features the first principal component is always the average direction, it always explains a fraction \((1+r)/2\) of the variance, and no computer is needed to know it. With \(r = 0.9792\) for MolWt and HeavyAtoms the eigenvalues must be 1.9792 and 0.0208, the first component must explain \((1+0.9792)/2 = 0.9896\) of the variance, and the loadings must both be \(1/\sqrt{2} = 0.7071\). The next chunk checks all four numbers.



In [ ]:
## Worked example: PCA of two standardised, correlated descriptors - by hand and by prcomp.
Zp  <- scale(P[, c("MolWt", "HeavyAtoms")])
rho <- cor(Zp)[1, 2]
pc2 <- prcomp(Zp)
cat(sprintf("correlation r = %.4f\n", rho))
cat(sprintf("by hand : eigenvalues 1+r = %.4f and 1-r = %.4f -> PC1 explains (1+r)/2 = %.4f\n",
            1 + rho, 1 - rho, (1 + rho) / 2))
cat(sprintf("prcomp  : eigenvalues %s -> PC1 explains %.4f\n",
            paste(round(pc2$sdev^2, 4), collapse = " and "), pc2$sdev[1]^2 / sum(pc2$sdev^2)))
cat("PC1 direction by hand (1,1)/sqrt(2) = 0.7071 0.7071 | prcomp:",
    round(abs(pc2$rotation[, 1]), 4), "\n")
Z   <- scale(P)
pca <- prcomp(Z, center = FALSE, scale. = FALSE)
sv  <- svd(Z)
cat("\nSVD identity s^2/(n-1) == prcomp variances on all 32 columns:",
    isTRUE(all.equal(sv$d^2 / (nrow(Z) - 1), pca$sdev^2)), "\n")



The hand calculation and `prcomp` agree to four decimals, and the identity \(s_j^2/(n-1) = \lambda_j\) holds on the full matrix. Two consequences follow immediately and are often forgotten. First, PCA on *unstandardised* columns is PCA of whatever units you happened to use — `heavy_sq` in squared atoms will dominate QED on a 0–1 scale — so standardise unless the columns are already commensurable; `prcomp(x, scale. = TRUE)` does it, and inside a cross-validation the centre and scale must come from the training fold. Second, the sign of a component is arbitrary: a column of `rotation` and its negative are the same direction, so never interpret the sign of a loading without fixing a convention.

### 4.2 Scree, loadings and the biplot



In [ ]:
## Scree plot and cumulative variance for the panel and for the fingerprints.
ve   <- pca$sdev^2 / sum(pca$sdev^2)
vefp <- pfp$sdev^2 / sum(pfp$sdev^2)
p1 <- ggplot(data.frame(k = 1:15, pct = 100 * ve[1:15], cum = 100 * cumsum(ve[1:15])), aes(k)) +
  geom_col(aes(y = pct), fill = PAL[["blue"]], width = 0.7) +
  geom_line(aes(y = cum), colour = PAL[["orange"]]) + geom_point(aes(y = cum), colour = PAL[["orange"]], size = 1.2) +
  labs(x = "component", y = "% of variance (bars) / cumulative % (line)",
       title = "Scree: 32-column panel")
p2 <- ggplot(data.frame(k = 1:60, cum = 100 * cumsum(vefp[1:60])), aes(k, cum)) +
  geom_line(colour = PAL[["green"]]) + geom_point(size = 1, colour = PAL[["green"]]) +
  ylim(0, 100) +
  labs(x = "number of components", y = "cumulative % of variance",
       title = "Fingerprints need far more components")
print(p1 | p2)
cat(sprintf("panel: PC1 %.1f%%, PC1+PC2 %.1f%%, components for 90%% of variance: %d of %d\n",
            100 * ve[1], 100 * sum(ve[1:2]), which(cumsum(ve) >= 0.90)[1], ncol(Z)))
cat(sprintf("fingerprints: PC1 %.1f%%, first 60 components %.1f%% of variance\n",
            100 * vefp[1], 100 * sum(vefp[1:60])))



<p class="caption"><b>Figure 5.</b> Left: the scree plot of the 32-column panel — the first component alone carries 34.6% of the variance (it is essentially "molecular size", as the loadings below confirm), the first two 55.6%, and six components reach 90%. Right: the cumulative curve for the 2,048 fingerprint bits, where the first component carries only 6.5% and sixty components reach 64.2%. Sparse binary data have no dominant directions, which is why PCA is a poor compressor for fingerprints and why the chemical-space maps in Section 5 are built from a non-linear embedding of those sixty components rather than from the components themselves. The practical rule: if the first two components explain less than about 30% of the variance, a two-dimensional PCA scatter plot is a shadow, not a summary<sup class="c"><a href="#r11">[11]</a></sup><sup class="c"><a href="#r16">[16]</a></sup>.</p>



In [ ]:
## Loadings and a biplot: what the first two components of the panel mean.
ld  <- as.data.table(pca$rotation[, 1:2], keep.rownames = "feature")
sc  <- pca$x
topl <- ld[order(-abs(PC1))][1:12][order(PC1)]
p1 <- ggplot(topl, aes(factor(feature, levels = topl$feature), PC1)) +
  geom_col(fill = PAL[["blue"]], width = 0.65) + coord_flip() +
  labs(x = NULL, y = "loading on PC1", title = "PC1 is molecular size") +
  theme(axis.text.y = element_text(size = 7.5))
arrows <- ld[feature %in% c("MolWt", "MolLogP", "TPSA", "FractionCSP3", "flexibility", "AromaticRings")]
p2 <- ggplot(data.frame(PC1 = sc[, 1], PC2 = sc[, 2], y = y), aes(PC1, PC2)) +
  geom_point(aes(colour = y), size = 0.7, alpha = 0.6) +
  scale_colour_viridis_c(name = "pActivity") +
  geom_segment(data = arrows, aes(x = 0, y = 0, xend = PC1 * 9, yend = PC2 * 9),
               arrow = arrow(length = unit(0.14, "cm")), colour = PAL[["orange"]], linewidth = 0.4,
               inherit.aes = FALSE) +
  geom_text(data = arrows, aes(PC1 * 10.2, PC2 * 10.2, label = feature),
            colour = PAL[["orange"]], size = 2.4, inherit.aes = FALSE) +
  labs(x = sprintf("PC1 (%.0f%%)", 100 * ve[1]), y = sprintf("PC2 (%.0f%%)", 100 * ve[2]),
       title = "Biplot, coloured by pActivity")
print(p1 | p2)
cat(sprintf("cor(PC1, MolWt) = %.3f | cor(PC1, pActivity) = %.3f\n", cor(sc[, 1], P[, "MolWt"]), cor(sc[, 1], y)))
cat(sprintf("cor(PC2, FractionCSP3) = %.3f | cor(PC2, AromaticRings) = %.3f\n",
            cor(sc[, 2], P[, "FractionCSP3"]), cor(sc[, 2], P[, "AromaticRings"])))



<p class="caption"><b>Figure 6.</b> Left: the twelve largest loadings on PC1 — heavy-atom count, molecular weight, its logarithm and its square, the drug-likeness score and the aromatic-ring/rotatable-bond interaction — all pointing the same way, which identifies PC1 as a size axis (its correlation with molecular weight is 0.932). Right: the biplot, with compounds as points coloured by potency and six panel columns as arrows. An arrow's direction shows how that column increases across the plane and its length how much of it the plane captures. PC2 separates aromatic from aliphatic character (correlation −0.308 with the aromatic-ring count against +0.144 with the sp³ fraction). The weak colour gradient is the honest message: potency is <em>not</em> a function of the first two principal components of bulk properties — PC1 correlates with pActivity at only 0.084 — which is exactly why fingerprint models beat descriptor models in Unit 11.</p>



In [ ]:
## Reconstruction: how much of the data survives a k-component compression?
rec_tbl <- rbindlist(lapply(c(1, 2, 3, 5, 10, 20, 32), function(k) {
  rec <- sc[, 1:k, drop = FALSE] %*% t(pca$rotation[, 1:k, drop = FALSE])
  data.table(components = k, variance_explained = sum(ve[1:k]),
             reconstruction_RMSE = sqrt(mean((Z - rec)^2)),
             ridge_R2_on_scores = cvR2(sc[, 1:k, drop = FALSE]))
}))
knitr::kable(rec_tbl[, lapply(.SD, function(v) if (is.numeric(v)) round(v, 3) else v)],
             caption = "PCA as a lossy compression of the standardised panel.")



<p class="caption"><b>Table (Section 4.2).</b> PCA as a lossy compression of the standardised panel. Ten components retain 97.6% of the variance and reconstruct the 32 columns with a root-mean-square error of 0.155 standard-deviation units; twenty retain 99.8% and reconstruct to 0.043. The last column is the one that matters for modelling, and it carries the warning: a ridge regression on the first <em>three</em> components — which keep 71% of the variance — reaches \(R^2\) = 0.006, indistinguishable from predicting the mean, and even ten components reach only 0.122. A compression that keeps three quarters of the <em>variance</em> can throw away essentially all of the <em>predictive information</em>. Variance is not relevance: PCA is unsupervised and has no reason to put the target-related directions first. (The last row is a curiosity worth a sentence: ridge on all 32 scores scores 0.163 against 0.128 on the 32 raw columns, because an orthogonal rotation changes how a single ridge penalty is distributed across directions — the models are not the same model.) Use PCA to compress, to visualise and to decorrelate, not as a free accuracy gain — and, being a fitted transformation, always inside the cross-validation.</p>

## 5. Non-linear embeddings: t-SNE and UMAP

PCA is a rotation: it preserves global structure and large distances, and it cannot unfold a curved manifold. Classical multidimensional scaling, which places points so that their map distances match a given distance matrix<sup class="c"><a href="#r17">[17]</a></sup>, has the same linear limitation when the distances are Euclidean. Two neighbour-embedding methods dominate practice instead.

- **t-SNE** converts distances into neighbour probabilities and finds a two-dimensional arrangement whose neighbour probabilities match, with a heavy-tailed kernel in the map that spreads moderately distant points apart. Its one essential hyperparameter, **perplexity**, is the effective number of neighbours each point attends to. It preserves local neighbourhoods and *destroys* global geometry: distances between clusters in a t-SNE map carry little information<sup class="c"><a href="#r18">[18]</a></sup><sup class="c"><a href="#r23">[23]</a></sup>.
- **UMAP** builds a fuzzy nearest-neighbour graph and optimises a low-dimensional layout of it<sup class="c"><a href="#r19">[19]</a></sup>. It is faster, retains somewhat more global structure, and can transform new points — but it has the same fundamental character: a neighbourhood-preserving projection whose large-scale features must not be over-read<sup class="c"><a href="#r20">[20]</a></sup><sup class="c"><a href="#r21">[21]</a></sup>.

Both are run here on the first 60 principal components of the fingerprint matrix, which is the standard recipe: reduce linearly to kill the noise and the cost, then embed. `Rtsne` needs `set.seed()` before the call (it has no seed argument), and `umap` is deterministic given the same seed and settings.



In [ ]:
## Chemical space of the 2,764 ROCK1 compounds: t-SNE and UMAP on 60 fingerprint PCs. (~30 s)
emb_in <- pfp$x
set.seed(0)
ts_xy <- Rtsne(emb_in, dims = 2, perplexity = 30, pca = FALSE, max_iter = 500,
               check_duplicates = FALSE, verbose = FALSE)$Y
set.seed(0)
um_xy <- umap(emb_in, n_neighbors = 15, min_dist = 0.1)$layout
emb <- rbindlist(list(
  data.table(method = "t-SNE (perplexity 30)", x = ts_xy[, 1], y2 = ts_xy[, 2], pAct = y, scaf = scaffold),
  data.table(method = "UMAP (15 neighbours)",  x = um_xy[, 1], y2 = um_xy[, 2], pAct = y, scaf = scaffold)))
print(ggplot(emb, aes(x, y2, colour = pAct)) + geom_point(size = 0.6, alpha = 0.65) +
  facet_wrap(~ method, scales = "free") + scale_colour_viridis_c(name = "pActivity") +
  labs(x = NULL, y = NULL, title = "Chemical space, coloured by potency") +
  theme(axis.text = element_blank(), panel.grid = element_blank()))
cat("embedded", nrow(emb_in), "compounds from", ncol(emb_in), "fingerprint principal components\n")



<p class="caption"><b>Figure 7.</b> The chemical space of the dataset as seen by two neighbour embeddings, coloured by potency. Both maps show many small islands rather than a continuum — the signature of a dataset built from discrete series — and both show potency varying <em>within</em> islands as well as between them, which is the geometric statement of the activity-cliff problem met in Unit 11. The two layouts are not comparable point by point: only the neighbourhood structure is meaningful, and the position, size and separation of the islands differ between the methods and between runs with different seeds.</p>



In [ ]:
## The same embeddings coloured by scaffold family: do the islands correspond to chemical series?
big <- names(sort(table(scaffold), decreasing = TRUE))[1:5]
emb[, family := ifelse(scaf %in% big, paste0("family ", match(scaf, big)), "other scaffolds")]
print(ggplot(emb[family == "other scaffolds"], aes(x, y2)) +
  geom_point(size = 0.4, colour = "grey80", alpha = 0.6) +
  geom_point(data = emb[family != "other scaffolds"], aes(colour = family), size = 0.9) +
  facet_wrap(~ method, scales = "free") +
  scale_colour_manual(values = unname(PAL[c("blue", "orange", "green", "purple")])[c(1, 2, 3, 4, 1)], name = NULL) +
  labs(x = NULL, y = NULL, title = "The islands are chemical series") +
  theme(axis.text = element_blank(), panel.grid = element_blank(), legend.position = "top"))
share_scaffold <- function(xy) {
  dm <- as.matrix(dist(xy)); diag(dm) <- Inf
  mean(vapply(seq_len(nrow(xy)), function(i) mean(scaffold[order(dm[i, ])[1:10]] == scaffold[i]), numeric(1)))
}
cat(sprintf("t-SNE: %.1f%% of each compound's 10 nearest map neighbours share its exact scaffold\n",
            100 * share_scaffold(ts_xy)))
cat(sprintf("UMAP : %.1f%%\n", 100 * share_scaffold(um_xy)))
cat("the five largest scaffold families cover", sum(scaffold %in% big),
    sprintf("compounds (%.0f%%)\n", 100 * mean(scaffold %in% big)))



<p class="caption"><b>Figure 8.</b> The same two maps with the five largest Bemis–Murcko scaffold families<sup class="c"><a href="#r34">[34]</a></sup> highlighted (380 compounds, 14% of the set). The families occupy coherent territories, confirming that the islands are chemical series rather than artefacts — and the printed diagnostic quantifies it: 29.5% (t-SNE) and 27.5% (UMAP) of each compound's ten nearest neighbours in the map share its <em>exact</em> scaffold, which is far above the chance level given 1,218 scaffolds of which 858 are singletons. This is the clearest picture of why the scaffold split of Unit 12 is hard: a random split samples points from inside every island, while a scaffold split removes whole islands.</p>

### 5.1 False clusters: the failure mode of neighbour embeddings

A neighbour embedding always produces a picture with structure, because the optimisation's job is to spread points apart. Give it data with no clusters at all and it will still draw some<sup class="c"><a href="#r22">[22]</a></sup><sup class="c"><a href="#r23">[23]</a></sup>. The experiment: 600 points of pure 30-dimensional Gaussian noise — one homogeneous blob by construction — embedded at three perplexities, with the apparent clustering quantified by the silhouette width of a six-cluster k-means fit in the map and in the original space.



In [ ]:
## t-SNE on structureless data: the clusters you see are made by the method. (~20 s)
set.seed(0); noise <- matrix(rnorm(600 * 30), nrow = 600)
set.seed(0); km0 <- kmeans(noise, 6, nstart = 5)
sil_orig <- mean(silhouette(km0$cluster, dist(noise))[, "sil_width"])
fake <- rbindlist(lapply(c(5, 30, 100), function(perp) {
  set.seed(0)
  xy <- Rtsne(noise, dims = 2, perplexity = perp, pca = FALSE, max_iter = 400, check_duplicates = FALSE)$Y
  set.seed(0); km <- kmeans(xy, 6, nstart = 5)
  s <- mean(silhouette(km$cluster, dist(xy))[, "sil_width"])
  cat(sprintf("perplexity %3d -> silhouette of 6 k-means clusters: %.3f in the map, %.3f in the original 30-D data\n",
              perp, s, sil_orig))
  data.table(panel = sprintf("perplexity %d (silhouette %.2f)", perp, s), x = xy[, 1], y2 = xy[, 2],
             cluster = factor(km$cluster))
}))
print(ggplot(fake, aes(x, y2, colour = cluster)) + geom_point(size = 0.6, alpha = 0.8, show.legend = FALSE) +
  facet_wrap(~ panel, scales = "free") +
  labs(x = NULL, y = NULL, title = "600 points of pure noise, embedded three times") +
  theme(axis.text = element_blank(), panel.grid = element_blank()))



<p class="caption"><b>Figure 9.</b> Six hundred points of pure Gaussian noise, embedded by t-SNE at three perplexities and coloured by a six-cluster k-means fit <em>of the map</em>. All three panels look like data with structure; the silhouette width of those clusters is 0.342 to 0.351 in the map against 0.029 in the original 30-dimensional space, where there is nothing. Low perplexity fragments the blob into more and tighter islands, high perplexity into fewer — the number of apparent clusters is a hyperparameter, not a property of the data. The discipline that follows: never cluster in the embedding and report the clusters as findings; cluster in the original space (or on the original distances, as Unit 15 does) and use the embedding only to display the result, and always show at least two perplexities or neighbour settings.</p>

<div class="exercise"><h4>Exercise 2</h4>
<p>Re-run Figure 7 with <code>perplexity = 5</code> and <code>perplexity = 150</code>, and UMAP with <code>n_neighbors = 5</code> and <code>n_neighbors = 100</code>. Which conclusions about the chemical space survive all four pictures? Write down the one sentence about the dataset that you would be willing to defend from these maps.</p></div>

## 6. The curse of dimensionality

In high dimension, volume grows so fast that every point is far from every other and — more damagingly — *all pairwise distances become similar*. Since nearest-neighbour methods, kernels, clustering and any similarity-based applicability domain depend on the *contrast* between near and far, their foundation dissolves as \(p\) grows<sup class="c"><a href="#r24">[24]</a></sup><sup class="c"><a href="#r25">[25]</a></sup>. The simulation below measures it directly: 500 Gaussian random points in \(d\) dimensions, the relative contrast \((\max - \min)/\min\) of their pairwise distances, and the coefficient of variation of those distances.



In [ ]:
## Distance concentration: the same 500 points, more and more dimensions.
set.seed(0)
conc <- rbindlist(lapply(c(2, 5, 10, 20, 50, 100, 500, 2000), function(dim) {
  dd <- dist(matrix(rnorm(500 * dim), nrow = 500))
  data.table(d = dim, min = min(dd), mean = mean(dd), max = max(dd),
             relative_contrast = (max(dd) - min(dd)) / min(dd), cv = sd(dd) / mean(dd))
}))
p1 <- ggplot(conc, aes(d, relative_contrast)) + geom_line(colour = PAL[["orange"]]) +
  geom_point(size = 1.3, colour = PAL[["orange"]]) + scale_x_log10() + scale_y_log10() +
  labs(x = "dimensions", y = "(max - min) / min distance", title = "Relative contrast collapses")
p2 <- ggplot(conc, aes(d, cv)) + geom_line(colour = PAL[["blue"]]) +
  geom_point(size = 1.3, colour = PAL[["blue"]]) + scale_x_log10() +
  labs(x = "dimensions", y = "sd / mean of pairwise distances", title = "All distances become the same")
print(p1 | p2)
knitr::kable(conc[, lapply(.SD, function(v) round(v, 4))],
             caption = "Pairwise-distance statistics of 500 random points as the dimension grows.")



<p class="caption"><b>Figure 10 and table.</b> Distance concentration measured on random Gaussian points. In two dimensions the farthest pair is nearly five thousand times further apart than the closest pair and the spread of distances is 52% of their mean; by 100 dimensions the relative contrast is 0.83 and the coefficient of variation 0.071; by 2,000 dimensions — the size of our fingerprint space — distances vary by 1.6% around their mean and the farthest pair is only 14% further than the closest. "Nearest neighbour" stops being a meaningful notion for isotropic high-dimensional data<sup class="c"><a href="#r24">[24]</a></sup>. Two caveats save practice: real data are not isotropic (they lie near a much lower-dimensional manifold, which is why 60 fingerprint components capture most of the structure), and the Tanimoto metric on sparse binary vectors concentrates far more slowly than Euclidean distance on dense ones — which is precisely why cheminformatics uses it<sup class="c"><a href="#r27">[27]</a></sup>. The general lesson stands: before using a distance in high dimension, check that it still discriminates<sup class="c"><a href="#r26">[26]</a></sup>.</p>

### 6.1 What it costs a real model

The simulation is abstract; the next chunk makes it concrete by appending columns of pure noise to the 32 standardised panel columns and re-measuring three models. Nothing is removed and no information is added — only the dimension grows.



In [ ]:
## Adding pure-noise columns to a real feature matrix: who survives? (~40 s)
f_knn <- function(Xtr, yy, Xte) predict(caret::knnreg(Xtr, yy, k = 10), Xte)
f_rf60 <- function(Xtr, yy, Xte) {
  m <- ranger(x = Xtr, y = yy, num.trees = 60, seed = 1, num.threads = 4); predict(m, Xte)$predictions
}
set.seed(0)
noise_tbl <- rbindlist(lapply(c(0, 50, 200, 1000), function(extra) {
  M <- if (extra) cbind(Z, matrix(rnorm(length(y) * extra), nrow = length(y))) else Z
  data.table(noise_columns = extra, total_columns = ncol(M),
             kNN10_R2 = cvR2(M, f_knn), forest_R2 = cvR2(M, f_rf60), ridge_R2 = cvR2(M))
}))
print(ggplot(melt(noise_tbl, id.vars = c("noise_columns", "total_columns")),
       aes(noise_columns, value, colour = variable)) +
  geom_hline(yintercept = 0, linetype = "dashed", linewidth = 0.4) +
  geom_line() + geom_point(size = 1.4) +
  scale_x_sqrt(breaks = c(0, 50, 200, 1000)) +
  scale_colour_manual(values = unname(PAL[c("orange", "blue", "green")]),
                      labels = c("10-nearest neighbours", "random forest", "ridge"), name = NULL) +
  labs(x = "columns of pure noise added to the 32 panel columns", y = expression(cross-validated~R^2),
       title = "The cost of irrelevant dimensions") + theme(legend.position = "top"))
knitr::kable(noise_tbl[, lapply(.SD, function(v) if (is.double(v)) round(v, 3) else v)],
             caption = "Three models as pure-noise columns are appended.")



<p class="caption"><b>Figure 11 and table.</b> The price of dimensions that carry nothing. The nearest-neighbour model, which depends entirely on distance contrast, falls from \(R^2\) = 0.273 to 0.090 with 50 noise columns and to −0.054 with 1,000: its neighbourhoods are now defined by noise. Ridge falls from 0.128 to −0.002 at 1,000 columns, because a fixed penalty cannot control 1,032 coefficients — the remedy would be to raise \(\lambda\), that is, to tune. The random forest degrades gracefully, from 0.338 to 0.113, because each split searches for a feature that actually reduces impurity and mostly ignores the rest; this robustness to irrelevant features is one of the reasons tree ensembles are the default for wide scientific tables (Unit 14).</p>

<div class="box key"><h4>Key idea</h4>
<p>Dimensionality is not just a computational cost, it changes the geometry. Added columns that carry no signal make every distance-based method worse, not merely slower — which is the real argument for feature selection, and the reason a model with 32 informative descriptors can beat one with 2,048 mostly empty bits on a small dataset while losing on a large one.</p></div>

<div class="box bridge"><h4>Bridge — where this unit sits</h4>
<p><a href="../U11_Supervised_Workflow/index.html">Unit 11 (R)</a> established why every fitted transformation in this unit lives inside the resampling; <a href="../U12_Validation/index.html">Unit 12 (R)</a> supplies the scaffold-grouped split under which the hashed-scaffold experiment of Section 2.4 was judged; <a href="../U14_Ensembles_Tuning/index.html">Unit 14 (R)</a> takes this same panel into tuned gradient boosting and shows why tree ensembles tolerate the irrelevant columns of Section 6.1. The <a href="../index.html">R-edition index</a> lists the remaining units — Unit 15 clusters the same chemical space in its original metric rather than in the embedding, Unit 16 replaces the importance bar charts of Section 3.1 with Shapley values, Unit 17 turns distance-to-training into a calibrated applicability domain, and Unit 21 treats the expression matrix of Section 3.2 as a biological question rather than a feature-selection exercise. The Python counterpart of this unit is <a href="../../Python/U13_Features_Dimensionality/index.html">here</a>. The chemistry of the two molecular representations is <a href="../../../Cheminformatics/U5_Descriptors_Fingerprints/index.html">Unit 5 of the cheminformatics course</a> and the chemical-space mapping <a href="../../../Cheminformatics/U8_Chemical_Space/index.html">its Unit 8</a>; the linear algebra behind the SVD and the machine-learning treatment of dimensionality reduction are in the artificial-intelligence course, whose R edition covers clustering and PCA in <a href="../../../BGM411/R/U5_Clustering_PCA_ICA/index.html">U5</a>.</p></div>

## 7. Summary

You should now be able to explain, without notes:

- why a feature matrix is a scientific claim, and how descriptors, fingerprints and expression matrices differ in rows per column, sparsity and correlation;
- how to construct domain features (ratios, densities, transformations, interactions, hashed categoricals) and why they often add less than genuinely new information;
- that identity-like features (scaffold, patient, batch) inflate random-split scores and collapse under a group split;
- the three families of selection, the rule that selection is refitted inside every fold, and the fact that selection usually costs a regularised model accuracy while buying interpretability and cost;
- how to measure selection stability with the Jaccard index, and why a 20-gene signature from 138 samples is a hypothesis rather than a result;
- PCA as the SVD of the centred matrix, including what the scree plot, loadings, biplot and reconstruction error each tell you, and why variance explained is not predictive information;
- what t-SNE and UMAP preserve, and how to recognise a false cluster;
- distance concentration and its consequences for neighbours, kernels and applicability domains.

The numbers to remember: the hashed-scaffold feature is worth +0.064 \(R^2\) under a random split and −0.204 under a scaffold-grouped split; selection stability is Jaccard 0.906 for 32 panel columns on 2,764 compounds and 0.354 for 500 genes on 138 samples, with only 4 of 20 genes selected in every fold while the classifier still reaches ROC-AUC 0.909; three principal components keep 71% of the panel's variance and almost none of its predictive value; and t-SNE assigns a silhouette width of about 0.35 to clusters that do not exist.

## 8. Self-assessment

<details><summary>1. Your model improves when you add a one-hot encoding of the assay identifier. Should you keep it?</summary>Only if the assay will be known for future predictions and you validate under a split that groups by assay. Otherwise it is an identity feature of the same kind as the hashed scaffold: +0.064 \(R^2\) under a random split, −0.204 under a grouped split.</details>

<details><summary>2. Why does feature selection not improve the ridge model on the 32-column panel?</summary>Because the L2 penalty already shrinks redundant and useless columns; removing them removes some information and adds a selection step that can only be estimated with error. Selection buys interpretability, measurement cost and stability, not accuracy, when the model is already regularised — here every selector scored below the full panel's 0.128.</details>

<details><summary>3. A collaborator selects the 50 most differentially expressed genes on the whole dataset, then cross-validates a classifier on those 50. What is wrong and how large is the effect?</summary>The selection saw the test labels, so the cross-validation is not blind; Unit 11 measured the same mistake on fingerprints (\(R^2\) +0.140 against −0.100 at n = 150, and +0.563 against −0.227 on pure noise). On \(p \gg n\) expression data the effect is at its largest.</details>

<details><summary>4. What does a Jaccard index of 0.354 across folds mean for a published signature?</summary>That two folds of the same dataset agree on about a third of the selected features — here 43 different genes entered a 20-gene signature across five folds. The signature is one of many equivalent subsets; report selection frequencies or use stability selection, and do not present the specific gene list as the discovery.</details>

<details><summary>5. PC1 explains 34.6% of the variance of your panel. Is it the most predictive direction?</summary>Not necessarily — PCA is unsupervised. Here PC1 is molecular size and correlates with pActivity at 0.084; three components keep 71% of the variance and give a ridge \(R^2\) of 0.006. Variance explained measures reconstruction, not relevance.</details>

<details><summary>6. Why standardise before PCA, and when should you not?</summary>Standardise when columns have different units or wildly different scales, otherwise the largest-variance column dominates the first component by accident — <code>heavy_sq</code> would swamp QED here. Do not standardise when all columns are already in the same meaningful unit (fingerprint bits, log-expression values), where relative variance is itself information.</details>

<details><summary>7. Two clusters in your t-SNE map are far apart. What can you conclude?</summary>Almost nothing about how different they are. t-SNE preserves neighbourhoods, not distances between groups; the gap can be an artefact of perplexity or initialisation. Verify structure in the original space, and show more than one hyperparameter setting.</details>

<details><summary>8. Why does PCA need sixty components on fingerprints and six on the descriptor panel?</summary>Because the panel columns are strongly correlated measures of a few underlying properties (mostly size), while fingerprint bits are sparse, nearly uncorrelated records of distinct substructures — median pairwise correlation 0.006 — so there is no dominant direction to find and variance is spread across hundreds of components.</details>

<details><summary>9. How does distance concentration affect an applicability domain built on Euclidean distance over 2,048 bits?</summary>It makes it nearly useless: in 2,000 isotropic dimensions all distances fall within a couple of percent of their mean (coefficient of variation 0.016 in the simulation), so no threshold separates "inside" from "outside". Use Tanimoto similarity on the sparse binary vectors, or a distance in a reduced space, and check that the chosen metric still discriminates.</details>

<details><summary>10. When is feature hashing preferable to one-hot encoding?</summary>When the number of levels is large, unbounded or unknown in advance (scaffolds, assay identifiers, text tokens). It gives a fixed width and handles unseen levels at the price of collisions — here 1,218 scaffolds compressed into 256 buckets, with 1,203 of them sharing a bucket and the most crowded holding 13.</details>

<details><summary>11. You must deliver a diagnostic panel of at most 10 genes. Which selection procedure do you use?</summary>A stability-based one: repeat the selection over many resamples and keep the genes chosen most often, then report cross-validated performance of the fixed 10-gene panel with the selection still performed inside the folds for the estimate. Expect a modest performance loss against the full matrix and a large gain in deliverability.</details>

<details><summary>12. Permutation importance gives two correlated columns near-zero importance each. What does that mean?</summary>That either one can replace the other: shuffling one leaves the signal intact through its partner, so neither looks necessary. It is a statement about redundancy, not irrelevance — which is also why our permutation importances sum to 0.262 while the model's held-out \(R^2\) is 0.338. Group the correlated block and permute it jointly, or use a method designed for correlated features.</details>

## Glossary

<dl class="gloss">
<dt>Biplot</dt><dd>A scatter plot of PCA scores with the feature loadings drawn as arrows in the same plane.</dd>
<dt>Curse of dimensionality</dt><dd>The loss of distance contrast and the explosion of required sample size as the number of features grows.</dd>
<dt>Embedded selection</dt><dd>Selection performed as part of model fitting, as by the L1 penalty of the lasso.</dd>
<dt>Feature hashing</dt><dd>Mapping categorical levels into a fixed number of buckets with a hash function, accepting collisions.</dd>
<dt>Filter selection</dt><dd>Model-independent ranking of features by a univariate score (variance, correlation, t or F statistic, mutual information).</dd>
<dt>Jaccard index</dt><dd>Size of the intersection over size of the union of two sets; used here to measure selection stability across folds.</dd>
<dt>Loading</dt><dd>The weight of an original feature in a principal component; the entries of the matrix V of the SVD (<code>prcomp()$rotation</code>).</dd>
<dt>Mutual information</dt><dd>An information-theoretic measure of dependence that detects non-linear as well as linear relationships.</dd>
<dt>Perplexity</dt><dd>t-SNE's hyperparameter setting the effective number of neighbours each point considers.</dd>
<dt>Permutation importance</dt><dd>The loss in held-out performance when one feature's values are shuffled.</dd>
<dt>p ≫ n</dt><dd>The regime with more features than samples, where models fit perfectly in sample and selections are unstable.</dd>
<dt>PCA</dt><dd>Projection onto the orthogonal directions of maximal variance; computed as the SVD of the centred data matrix.</dd>
<dt>Reconstruction error</dt><dd>The distance between the original data and their projection back from a truncated set of components.</dd>
<dt>Recursive feature elimination</dt><dd>Fit, rank, drop the weakest features, repeat; the standard wrapper method.</dd>
<dt>Scree plot</dt><dd>Variance explained against component number; used to choose how many components to keep.</dd>
<dt>Silhouette width</dt><dd>How much closer a point is to its own cluster than to the next nearest; used here to show that map clusters are artefacts.</dd>
<dt>Stability selection</dt><dd>Repeating a selection over resamples and keeping features chosen with high frequency.</dd>
<dt>t-SNE / UMAP</dt><dd>Neighbour-embedding methods that preserve local structure for visualisation and distort global geometry.</dd>
</dl>

## References and further reading

<ol class="refs">
<li id="r1">Guyon I, Elisseeff A. An introduction to feature extraction. In: <em>Feature Extraction: Foundations and Applications</em>. Studies in Fuzziness and Soft Computing, vol 207. Springer; 2006:1-25. <a href="https://doi.org/10.1007/978-3-540-35488-8_1">doi:10.1007/978-3-540-35488-8_1</a></li>
<li id="r2">Saeys Y, Inza I, Larrañaga P. A review of feature selection techniques in bioinformatics. <em>Bioinformatics</em>. 2007;23(19):2507-2517. <a href="https://doi.org/10.1093/bioinformatics/btm344">doi:10.1093/bioinformatics/btm344</a></li>
<li id="r3">Ambroise C, McLachlan GJ. Selection bias in gene extraction on the basis of microarray gene-expression data. <em>Proc Natl Acad Sci USA</em>. 2002;99(10):6562-6566. <a href="https://doi.org/10.1073/pnas.102102699">doi:10.1073/pnas.102102699</a></li>
<li id="r4">Meinshausen N, Bühlmann P. Stability selection. <em>J R Stat Soc Series B</em>. 2010;72(4):417-473. <a href="https://doi.org/10.1111/j.1467-9868.2010.00740.x">doi:10.1111/j.1467-9868.2010.00740.x</a></li>
<li id="r5">Kalousis A, Prados J, Hilario M. Stability of feature selection algorithms: a study on high-dimensional spaces. <em>Knowl Inf Syst</em>. 2007;12:95-116. <a href="https://doi.org/10.1007/s10115-006-0040-8">doi:10.1007/s10115-006-0040-8</a></li>
<li id="r6">Guyon I, Weston J, Barnhill S, Vapnik V. Gene selection for cancer classification using support vector machines. <em>Mach Learn</em>. 2002;46:389-422. <a href="https://doi.org/10.1023/A:1012487302797">doi:10.1023/A:1012487302797</a></li>
<li id="r7">Tibshirani R. Regression shrinkage and selection via the lasso. <em>J R Stat Soc Series B</em>. 1996;58(1):267-288. <a href="https://doi.org/10.1111/j.2517-6161.1996.tb02080.x">doi:10.1111/j.2517-6161.1996.tb02080.x</a></li>
<li id="r8">Zou H, Hastie T. Regularization and variable selection via the elastic net. <em>J R Stat Soc Series B</em>. 2005;67(2):301-320. <a href="https://doi.org/10.1111/j.1467-9868.2005.00503.x">doi:10.1111/j.1467-9868.2005.00503.x</a></li>
<li id="r9">Altmann A, Toloşi L, Sander O, Lengauer T. Permutation importance: a corrected feature importance measure. <em>Bioinformatics</em>. 2010;26(10):1340-1347. <a href="https://doi.org/10.1093/bioinformatics/btq134">doi:10.1093/bioinformatics/btq134</a></li>
<li id="r10">Strobl C, Boulesteix AL, Zeileis A, Hothorn T. Bias in random forest variable importance measures: illustrations, sources and a solution. <em>BMC Bioinformatics</em>. 2007;8:25. <a href="https://doi.org/10.1186/1471-2105-8-25">doi:10.1186/1471-2105-8-25</a></li>
<li id="r11">Jolliffe IT, Cadima J. Principal component analysis: a review and recent developments. <em>Philos Trans R Soc A</em>. 2016;374:20150202. <a href="https://doi.org/10.1098/rsta.2015.0202">doi:10.1098/rsta.2015.0202</a></li>
<li id="r12">Pearson K. On lines and planes of closest fit to systems of points in space. <em>Philos Mag</em>. 1901;2(11):559-572. <a href="https://doi.org/10.1080/14786440109462720">doi:10.1080/14786440109462720</a></li>
<li id="r13">Hotelling H. Analysis of a complex of statistical variables into principal components. <em>J Educ Psychol</em>. 1933;24(6):417-441. <a href="https://doi.org/10.1037/h0071325">doi:10.1037/h0071325</a></li>
<li id="r14">Eckart C, Young G. The approximation of one matrix by another of lower rank. <em>Psychometrika</em>. 1936;1:211-218. <a href="https://doi.org/10.1007/BF02288367">doi:10.1007/BF02288367</a></li>
<li id="r15">Halko N, Martinsson PG, Tropp JA. Finding structure with randomness: probabilistic algorithms for constructing approximate matrix decompositions. <em>SIAM Rev</em>. 2011;53(2):217-288. <a href="https://doi.org/10.1137/090771806">doi:10.1137/090771806</a></li>
<li id="r16">Lever J, Krzywinski M, Altman N. Principal component analysis. <em>Nat Methods</em>. 2017;14:641-642. <a href="https://doi.org/10.1038/nmeth.4346">doi:10.1038/nmeth.4346</a></li>
<li id="r17">Kruskal JB. Multidimensional scaling by optimizing goodness of fit to a nonmetric hypothesis. <em>Psychometrika</em>. 1964;29:1-27. <a href="https://doi.org/10.1007/BF02289565">doi:10.1007/BF02289565</a></li>
<li id="r18">van der Maaten L, Hinton G. Visualizing data using t-SNE. <em>J Mach Learn Res</em>. 2008;9:2579-2605. <a href="https://www.jmlr.org/papers/v9/vandermaaten08a.html">jmlr.org/papers/v9/vandermaaten08a.html</a> (JMLR assigns no DOI; the R interface is the <code>Rtsne</code> package)</li>
<li id="r19">McInnes L, Healy J, Melville J. UMAP: uniform manifold approximation and projection for dimension reduction. <em>arXiv</em>. 2018. <a href="https://doi.org/10.48550/arXiv.1802.03426">doi:10.48550/arXiv.1802.03426</a></li>
<li id="r20">Kobak D, Berens P. The art of using t-SNE for single-cell transcriptomics. <em>Nat Commun</em>. 2019;10:5416. <a href="https://doi.org/10.1038/s41467-019-13056-x">doi:10.1038/s41467-019-13056-x</a></li>
<li id="r21">Becht E, McInnes L, Healy J, et al. Dimensionality reduction for visualizing single-cell data using UMAP. <em>Nat Biotechnol</em>. 2019;37:38-44. <a href="https://doi.org/10.1038/nbt.4314">doi:10.1038/nbt.4314</a></li>
<li id="r22">Chari T, Pachter L. The specious art of single-cell genomics. <em>PLoS Comput Biol</em>. 2023;19(8):e1011288. <a href="https://doi.org/10.1371/journal.pcbi.1011288">doi:10.1371/journal.pcbi.1011288</a></li>
<li id="r23">Wattenberg M, Viégas F, Johnson I. How to use t-SNE effectively. <em>Distill</em>. 2016;1(10):e2. <a href="https://doi.org/10.23915/distill.00002">doi:10.23915/distill.00002</a></li>
<li id="r24">Beyer K, Goldstein J, Ramakrishnan R, Shaft U. When is "nearest neighbor" meaningful? In: <em>ICDT 1999</em>. LNCS 1540. Springer; 1999:217-235. <a href="https://doi.org/10.1007/3-540-49257-7_15">doi:10.1007/3-540-49257-7_15</a></li>
<li id="r25">Aggarwal CC, Hinneburg A, Keim DA. On the surprising behavior of distance metrics in high dimensional space. In: <em>ICDT 2001</em>. LNCS 1973. Springer; 2001:420-434. <a href="https://doi.org/10.1007/3-540-44503-X_27">doi:10.1007/3-540-44503-X_27</a></li>
<li id="r26">Domingos P. A few useful things to know about machine learning. <em>Commun ACM</em>. 2012;55(10):78-87. <a href="https://doi.org/10.1145/2347736.2347755">doi:10.1145/2347736.2347755</a></li>
<li id="r27">Rogers D, Hahn M. Extended-connectivity fingerprints. <em>J Chem Inf Model</em>. 2010;50(5):742-754. <a href="https://doi.org/10.1021/ci100050t">doi:10.1021/ci100050t</a></li>
<li id="r28">Todeschini R, Consonni V. <em>Molecular Descriptors for Chemoinformatics</em>. 2nd ed. Wiley-VCH; 2009. <a href="https://doi.org/10.1002/9783527628766">doi:10.1002/9783527628766</a></li>
<li id="r29">Weinberger K, Dasgupta A, Langford J, Smola A, Attenberg J. Feature hashing for large scale multitask learning. In: <em>Proc. 26th Int. Conf. on Machine Learning (ICML '09)</em>. ACM; 2009:1113-1120. <a href="https://doi.org/10.1145/1553374.1553516">doi:10.1145/1553374.1553516</a></li>
<li id="r30">Papapanou PN, Behle JH, Kebschull M, et al. Subgingival bacterial colonization profiles correlate with gingival tissue gene expression. <em>BMC Microbiol</em>. 2009;9:221. <a href="https://doi.org/10.1186/1471-2180-9-221">doi:10.1186/1471-2180-9-221</a></li>
<li id="r31">Kebschull M, Demmer RT, Grün B, Guarnieri P, Pavlidis P, Papapanou PN. Gingival tissue transcriptomes identify distinct periodontitis phenotypes. <em>J Dent Res</em>. 2014;93(5):459-468. <a href="https://doi.org/10.1177/0022034514527288">doi:10.1177/0022034514527288</a></li>
<li id="r32">Leek JT, Scharpf RB, Bravo HC, et al. Tackling the widespread and critical impact of batch effects in high-throughput data. <em>Nat Rev Genet</em>. 2010;11:733-739. <a href="https://doi.org/10.1038/nrg2825">doi:10.1038/nrg2825</a></li>
<li id="r33">Box GEP, Cox DR. An analysis of transformations. <em>J R Stat Soc Series B</em>. 1964;26(2):211-243. <a href="https://doi.org/10.1111/j.2517-6161.1964.tb00553.x">doi:10.1111/j.2517-6161.1964.tb00553.x</a></li>
<li id="r34">Bemis GW, Murcko MA. The properties of known drugs. 1. Molecular frameworks. <em>J Med Chem</em>. 1996;39(15):2887-2893. <a href="https://doi.org/10.1021/jm9602928">doi:10.1021/jm9602928</a></li>
<li id="r35">Kuhn M. Building predictive models in R using the caret package. <em>J Stat Softw</em>. 2008;28(5):1-26. <a href="https://doi.org/10.18637/jss.v028.i05">doi:10.18637/jss.v028.i05</a></li>
<li id="r36">Friedman J, Hastie T, Tibshirani R. Regularization paths for generalized linear models via coordinate descent. <em>J Stat Softw</em>. 2010;33(1):1-22. <a href="https://doi.org/10.18637/jss.v033.i01">doi:10.18637/jss.v033.i01</a></li>
</ol>

<h4>Textbooks</h4>
<ul>
<li>Hastie T, Tibshirani R, Friedman J. <em>The Elements of Statistical Learning</em>. 2nd ed. Springer; 2009 (free PDF). — Chapters 3 (shrinkage and selection), 14 (unsupervised learning, PCA) and 18 (\(p \gg n\)).</li>
<li>James G, Witten D, Hastie T, Tibshirani R. <em>An Introduction to Statistical Learning with Applications in R</em>. 2nd ed. Springer; 2021 (free PDF). — Chapters 6 and 12 at an introductory level, with R code.</li>
<li>Todeschini R, Consonni V. <em>Molecular Descriptors for Chemoinformatics</em>. Wiley-VCH; 2009. — The reference work on what each descriptor measures.</li>
<li>Kuhn M, Johnson K. <em>Feature Engineering and Selection: A Practical Approach for Predictive Models</em>. CRC Press; 2019 (free online). — The R-centred treatment of everything in Sections 2 and 3.</li>
</ul>

<h4>Online resources</h4>
<ul>
<li><code>caret</code> documentation, "Pre-Processing" and "Feature Selection": <a href="https://topepo.github.io/caret/">topepo.github.io/caret</a>.</li>
<li>UMAP documentation, including "Understanding UMAP" and the parameter guide (the R package follows the same parameters): <a href="https://umap-learn.readthedocs.io/">umap-learn.readthedocs.io</a>.</li>
<li>TeachOpenCADD talktorial T005 (compound clustering) and T006 (maximum common substructures), which map the same chemical space: <a href="https://projects.volkamerlab.org/teachopencadd/">projects.volkamerlab.org/teachopencadd</a>.</li>
<li>Practical Cheminformatics (P. Walters) — posts on descriptors, chemical-space visualisation and the perils of t-SNE: <a href="https://patwalters.github.io/practicalcheminformatics/">patwalters.github.io/practicalcheminformatics</a>.</li>
</ul>
